# EKBE: purchase order history (EDA)

One row per PO history event: goods receipt (GR, category `E`) or invoice receipt (IR, category `Q`).
**Not needed for Q1–Q4**, which are all about *ordered* value (EKPO). This notebook checks whether
EKBE could support the "what extra data would Antonio need" angle (delivery reliability, invoiced
vs ordered).

Raw export: 2,824 rows x 93 columns; `sap_data.load` keeps 15.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # src/, for sap_data

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sap_data import load, load_raw, PURCH_ORG, COMPANY_CODE, PERIOD_START, PERIOD_END

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

In [ ]:
ekbe = load("ekbe")
ekpo = load("ekpo")
ekko = load("ekko")
print(ekbe.shape)
display(ekbe.head())
ekbe.dtypes

## 1. Grain and link to EKPO

In [ ]:
key = ["Purchasing Document", "Item", "Trans./event type",
       "Material Doc. Year", "Material Document", "Material Doc.Item"]
assert not ekbe.duplicated(key).any()
item_idx = ekpo.set_index(["Purchasing Document", "Item"]).index
print("History rows without EKPO item:",
      (~ekbe.set_index(["Purchasing Document", "Item"]).index.isin(item_idx)).sum())

> **Note.** Grain is one row per item × material/accounting document. All but one history row link to
> an EKPO item.

## 2. Event types

In [ ]:
display(pd.crosstab(ekbe["PO History Category"], ekbe["Movement Type"].fillna(0).astype(int),
                    margins=True))
display(pd.crosstab(ekbe["PO History Category"], ekbe["Debit/Credit ind"]))
display(ekbe["Currency"].value_counts(dropna=False))
ekbe.groupby("PO History Category")[["Quantity", "Amount"]].sum()

> **Note.** `E` = goods receipts (movement 101, plus 2 reversals 102); `Q` = invoice receipts.
> `H` (credit) rows are reversals or credit memos and must be subtracted if EKBE is ever summed.
> Almost all USD.

## 3. Coverage of PO items

In [ ]:
pf = ekpo[(ekpo["Purch. Doc. Category"] == "F") & ekpo["Deletion Indicator"].isna()]
pf_idx = pf.set_index(["Purchasing Document", "Item"]).index
for cat, label in [("E", "goods receipt"), ("Q", "invoice")]:
    has = ekbe.loc[ekbe["PO History Category"] == cat].set_index(["Purchasing Document", "Item"]).index
    print(f"PO items with a {label}: {pf_idx.isin(has).sum()} of {len(pf_idx)} "
          f"({pf_idx.isin(has).mean():.0%})")

## 4. Could EKBE measure delivery performance?

In [ ]:
gr = (ekbe[ekbe["PO History Category"] == "E"]
      .merge(ekko[["Purchasing Document", "Document Date"]].rename(columns={"Document Date": "PO Date"}),
             on="Purchasing Document", how="left", validate="many_to_one"))
gr["days_po_to_gr"] = (gr["Posting Date"] - gr["PO Date"]).dt.days
display(gr["days_po_to_gr"].describe())
gr["days_po_to_gr"].clip(-10, 30).value_counts().sort_index().head(15)

> **Note.** The median time from PO to goods receipt is **0 days**: students post the GR right after
> creating the PO, sometimes before it (negative lags). EKBE also has no *planned* delivery date (that
> lives in EKET, which wasn't extracted). EKBE therefore **can't** support a delivery-reliability
> claim. That's the point to make in the limitations: Antonio would need real GR dates vs. EKET
> scheduled dates, plus quality data (QM inspection lots / rejection rates) and invoice vs. PO price
> variances before changing a supplier relationship.

## Summary for the report (EKBE)
- Not used for Q1–Q4 (ordered value comes from EKPO).
- 71% of PO items have a GR and 57% an invoice, so "invoiced spend" would differ materially from "Net
  PO value". Keep the label "Net PO value".
- PO→GR lags are artificial (median 0 days), so no lead-time / reliability analysis is possible from
  this extract.